# Bloqade native simulator smoke test

Same 2-qubit circuit as `test.ipynb` (H on q0 → CX → X on q1 → Z on q0), but
written as a Bloqade QASM2 kernel and run on Bloqade's built-in
`StackMemorySimulator` (PyQrack-backed).

**Expected:** H + CX produces a Bell state {`|00⟩`, `|11⟩`}, X on q1 flips it
to {`|01⟩`, `|10⟩`}, and Z on q0 adds only an unobservable phase. Repeated
shots should split ~50/50 between `01` and `10`.

In [ ]:
from collections import Counter

from bloqade import qasm2
from bloqade.pyqrack import StackMemorySimulator

In [ ]:
def draw_bloqade(kernel, mode='text'):
    from qiskit import QuantumCircuit
    from bloqade.qasm2.emit import QASM2
    qc = QuantumCircuit.from_qasm_str(QASM2().emit_str(kernel))
    return qc.draw(output=mode)


## Build the Bloqade kernel

Bloqade uses the `@qasm2.main` decorator to turn a Python function into a
Kirin IR kernel. Inside the function we allocate quantum/classical registers
via `qasm2.qreg(n)` / `qasm2.creg(n)` and apply gates from the `qasm2`
namespace (`qasm2.h`, `qasm2.cx`, etc.).

In [ ]:
@qasm2.main
def circ():
    q = qasm2.qreg(2)
    c = qasm2.creg(2)
    qasm2.h(q[0])
    qasm2.cx(q[0], q[1])
    qasm2.x(q[1])
    qasm2.z(q[0])
    qasm2.measure(q, c)
    return c

print('Kernel:')
circ.print()

print(draw_bloqade(circ))                # ASCII
# draw_bloqade(circ, mode='mpl')         # matplotlib figure

## Single shot

`sim.run(kernel)` returns a `CRegister` whose entries are
`MeasurementResultValue` enums (0 or 1).

In [ ]:
sim = StackMemorySimulator(min_qubits=2)

creg = sim.run(circ)
bits = ''.join(str(int(b)) for b in creg)
print(f'single-shot result (q0 q1): {bits}')

## Many shots → histogram

Call `sim.run(circ)` repeatedly and tally outcomes. Expect the distribution
to concentrate on `01` and `10` in roughly equal proportion.

In [ ]:
shots = 1024
counts = Counter()
for _ in range(shots):
    creg = sim.run(circ)
    bits = ''.join(str(int(b)) for b in creg)
    counts[bits] += 1

print(f'{shots} shots:')
for outcome in sorted(counts):
    pct = 100.0 * counts[outcome] / shots
    print(f'  {outcome}: {counts[outcome]:>5d}  ({pct:5.1f}%)')

# Sanity check: only 01 and 10 should appear (up to shot noise).
expected = {'01', '10'}
unexpected = set(counts) - expected
assert not unexpected, f'unexpected outcomes appeared: {unexpected}'
print('\nOK — only expected outcomes (01, 10) observed.')

In [1]:
from bloqade import qasm2 as b_qasm2
from qiskit import QuantumCircuit
from qiskit import qasm2 as q_qasm2


qc = QuantumCircuit(3, 3)
qc.x(0)
qc.x(1)
qc.ccx(0, 1, 2)
qc.measure([0, 1, 2], [0, 1, 2])
print(qc.draw(output='text'))

q_qasm2.dump(qc, 'test.qasm')

kernel = b_qasm2.loadfile('test.qasm', returns="c")

     ┌───┐     ┌─┐      
q_0: ┤ X ├──■──┤M├──────
     ├───┤  │  └╥┘┌─┐   
q_1: ┤ X ├──■───╫─┤M├───
     └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ─────┤ X ├─╫──╫─┤M├
          └───┘ ║  ║ └╥┘
c: 3/═══════════╩══╩══╩═
                0  1  2 


In [2]:
import sys, os
sys.path.append(os.path.abspath('src'))
from run_bloqade import run_bloqade_simulation
from bloqade.pyqrack import StackMemorySimulator

run_bloqade_simulation(StackMemorySimulator(min_qubits=2), kernel, shots=1024)

[[<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>,
  <MeasurementResultValue.One: 1>],
 [<MeasurementResultValue.One: 1>,
  <Measurem